In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import pickle
import joblib

In [15]:
# Dataframe Importing

df = pd.read_csv('Reviews.csv')

df = df.drop(columns = ['Id','ProductId','UserId','ProfileName','HelpfulnessNumerator','HelpfulnessDenominator','Time'])

In [16]:
df.isnull().sum()

df = df.fillna('')

df

,Score,Summary,Text
0,5,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,4,"""Delight"" says it all",This is a confection that has been around a fe...
3,2,Cough Medicine,If you are looking for the secret ingredient i...
4,5,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...
568449,5,Will not do without,Great for sesame chicken..this is a good if no...
568450,2,disappointed,I'm disappointed with the flavor. The chocolat...
568451,5,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568452,5,Favorite Training and reward treat,These are the BEST treats for training and rew...


In [17]:
X = df.drop(columns = ['Score','Summary'])
y = df['Score']

print(X)
print("\n-------------------------------------------------------------\n")
print(y)

                                                     Text
0       I have bought several of the Vitality canned d...
1       Product arrived labeled as Jumbo Salted Peanut...
2       This is a confection that has been around a fe...
3       If you are looking for the secret ingredient i...
4       Great taffy at a great price.  There was a wid...
...                                                   ...
568449  Great for sesame chicken..this is a good if no...
568450  I'm disappointed with the flavor. The chocolat...
568451  These stars are small, so you can give 10-15 o...
568452  These are the BEST treats for training and rew...
568453  I am very satisfied ,product is as advertised,...

[568454 rows x 1 columns]

-------------------------------------------------------------

0         5
1         1
2         4
3         2
4         5
         ..
568449    5
568450    2
568451    5
568452    5
568453    5
Name: Score, Length: 568454, dtype: int64


In [18]:
X = X.drop_duplicates(subset=['Text'], keep='first')

X.duplicated().sum()

np.int64(0)

In [19]:
# Classifyng scores in a better way

def get_sentiment(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"

y = y.apply(get_sentiment)

y

0         positive
1         negative
2         positive
3         negative
4         positive
            ...   
568449    positive
568450    negative
568451    positive
568452    positive
568453    positive
Name: Score, Length: 568454, dtype: object

In [20]:
print("Original class distribution:")
print(y.value_counts())
print("\nPercentages:")
print(y.value_counts(normalize=True) * 100)

Original class distribution:
Score
positive    443777
negative     82037
neutral      42640
Name: count, dtype: int64

Percentages:
Score
positive    78.067355
negative    14.431599
neutral      7.501047
Name: proportion, dtype: float64


In [21]:
from sklearn.utils import resample

# ===== Extract text =====
X_text = X['Text'] if isinstance(X, pd.DataFrame) else X

# ===== Create combined DataFrame =====
df_combined = pd.DataFrame({
    'text': X_text,
    'sentiment': y
})

print("Original distribution:")
print(df_combined['sentiment'].value_counts())

# ===== Check and handle empty classes =====
df_positive = df_combined[df_combined['sentiment'] == 'positive']
df_negative = df_combined[df_combined['sentiment'] == 'negative']
df_neutral = df_combined[df_combined['sentiment'] == 'neutral']

print(f"\nClass sizes:")
print(f"Positive: {len(df_positive)}")
print(f"Negative: {len(df_negative)}")
print(f"Neutral: {len(df_neutral)}")

# If neutral is empty, we need to fix it
if len(df_neutral) == 0:
    print("\n⚠️ Neutral class is empty! Checking if there are any score=3 values...")
    # Check if there are any score=3 in original y
    score_3_count = (y == 3).sum()
    print(f"Number of score=3 in original data: {score_3_count}")
    
    if score_3_count > 0:
        print("There are score=3 values but they weren't properly processed.")
        print("Let's recreate sentiment labels to ensure neutral class is captured.")
        
        # Make sure y is numeric
        if y.dtype == 'object':
            y = pd.to_numeric(y, errors='coerce')
        
        # Recreate sentiment labels
        y_sentiment = y.apply(get_sentiment)
        df_combined = pd.DataFrame({
            'text': X_text,
            'sentiment': y_sentiment
        })
        
        # Check again
        df_neutral = df_combined[df_combined['sentiment'] == 'neutral']
        print(f"Neutral class size after fix: {len(df_neutral)}")

# Now proceed with balancing only if all classes have samples
if len(df_positive) > 0 and len(df_negative) > 0 and len(df_neutral) > 0:
    # Ternary classification
    max_size = max(len(df_positive), len(df_negative), len(df_neutral))
    print(f"\nUpsampling all classes to {max_size} samples...")
    
    df_positive_upsampled = resample(df_positive, replace=True, n_samples=max_size, random_state=42)
    df_negative_upsampled = resample(df_negative, replace=True, n_samples=max_size, random_state=42)
    df_neutral_upsampled = resample(df_neutral, replace=True, n_samples=max_size, random_state=42)
    
    df_balanced = pd.concat([df_positive_upsampled, df_negative_upsampled, df_neutral_upsampled])
    
elif len(df_positive) > 0 and len(df_negative) > 0:
    # Binary classification (no neutral)
    print("\nPerforming binary classification (positive vs negative)...")
    max_size = max(len(df_positive), len(df_negative))
    
    df_positive_upsampled = resample(df_positive, replace=True, n_samples=max_size, random_state=42)
    df_negative_upsampled = resample(df_negative, replace=True, n_samples=max_size, random_state=42)
    
    df_balanced = pd.concat([df_positive_upsampled, df_negative_upsampled])
    
else:
    raise ValueError("Insufficient data for classification")

# Shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Extract balanced data
X_balanced = df_balanced['text']
y_balanced = df_balanced['sentiment']

print("\n" + "="*50)
print("Balanced class distribution:")
print(y_balanced.value_counts())
print(f"\nBalanced dataset shape: {df_balanced.shape}")

Original distribution:
sentiment
positive    443777
negative     82037
neutral      42640
Name: count, dtype: int64

Class sizes:
Positive: 443777
Negative: 82037
Neutral: 42640

Upsampling all classes to 443777 samples...

Balanced class distribution:
sentiment
neutral     443777
negative    443777
positive    443777
Name: count, dtype: int64

Balanced dataset shape: (1331331, 2)


In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_balanced = le.fit_transform(y_balanced)

y_balanced

joblib.dump(le, 'label_encoder.pkl')
print("✅ label_encoder.pkl saved successfully!")

✅ label_encoder.pkl saved successfully!


In [23]:
X = X_balanced
y = y_balanced

# y = pd.Series(y)

print(X)
print("\n------\n")
print(y)

0                                                        NaN
1                                                        NaN
2          OK, first thing first. This coffee is cheap- a...
3          I gave the soup 2 stars instead of 1 becuse I ...
4          Tastes great. Seems to have slightly less fizz...
                                 ...                        
1331326    These cherries are perfect for just about anyt...
1331327    I am one of those fishermen who like sardines ...
1331328    Betty is a lab mix rescue dog. She is a very v...
1331329                                                  NaN
1331330                                                  NaN
Name: text, Length: 1331331, dtype: object

------

[1 0 1 ... 2 0 2]


In [24]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# nltk.download('stopwords')
# nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Safe preprocessing that handles any input type"""
    # Convert to string and handle NaN
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Remove HTML tags
    text = re.sub('<.*?>', '', text)
    
    # Keep only alphabetic characters and spaces
    text = re.sub('[^a-zA-Z]', ' ', text)
    
    # Split and process words
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w and w not in stop_words]
    
    return " ".join(words)

# Apply preprocessing with progress indicator
print("Preprocessing text...")
from tqdm import tqdm
tqdm.pandas()

# Method 1: Using progress_apply if you have tqdm
try:
    X_processed = X.progress_apply(preprocess_text)
except:
    # Method 2: Regular apply if tqdm not available
    X_processed = X.apply(preprocess_text)

# Remove empty strings
valid_mask = X_processed != ''
X_final = X_processed[valid_mask]

print(f"Kept {len(X_final):,} out of {len(X):,} samples ({len(X_final)/len(X)*100:.1f}%)")

Preprocessing text...


100%|██████████████████████████████████████████████████████████████████████| 1331331/1331331 [06:32<00:00, 3394.42it/s]


Kept 925,773 out of 1,331,331 samples (69.5%)


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,          # Keep top 5000 features
    stop_words='english',       # Remove English stopwords
    ngram_range=(1, 2),         # Use unigrams and bigrams
    min_df=2,                   # Ignore terms that appear in less than 2 documents
    max_df=0.95,                # Ignore terms that appear in more than 95% of documents
    sublinear_tf=True           # Use sublinear TF scaling (1+log(tf))
)
X = vectorizer.fit_transform(X_processed)

# Save vectorizer
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("✅ tfidf_vectorizer.pkl saved")

✅ tfidf_vectorizer.pkl saved


In [26]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size = 0.3,random_state = 42
)

Model Training

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

In [29]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
import time
import pandas as pd  # ADD THIS IMPORT
import joblib  # ADD THIS IMPORT

# Define models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        C=1.0,
        random_state=42
    ),
    
    'Linear SVM (SGD)': SGDClassifier(
        loss='hinge',
        alpha=0.0001,
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    ),
    
    'Multinomial NB': MultinomialNB(alpha=1.0),
    
    'Linear SVC': LinearSVC(
        C=1.0,
        max_iter=2000,
        random_state=42,
        class_weight='balanced'
    )
}

results = []
best_model = None  # Initialize best_model
best_f1 = 0  # Track best F1 score

print("\n" + "="*60)
print("MODEL COMPARISON FOR SENTIMENT ANALYSIS")
print("="*60)
print(f"Training samples: {X_train.shape[0]:,}")
print(f"Test samples: {X_test.shape[0]:,}")
print(f"Features: {X_train.shape[1]:,}")
print(f"Classes: {list(le.classes_)}")

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")
    
    try:
        # Training time
        start = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start
        
        # Prediction time
        start = time.time()
        y_pred = model.predict(X_test)
        predict_time = time.time() - start
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro')
        f1_weighted = f1_score(y_test, y_pred, average='weighted')
        
        # Cross-validation
        cv_scores = cross_val_score(
            model, X_train, y_train, 
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
            scoring='accuracy',
            n_jobs=-1
        )
        
        print(f"✓ Accuracy: {accuracy:.4f}")
        print(f"✓ F1-Macro: {f1_macro:.4f}")
        print(f"✓ F1-Weighted: {f1_weighted:.4f}")
        print(f"✓ CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
        print(f"✓ Train Time: {train_time:.2f}s")
        print(f"✓ Predict Time: {predict_time:.2f}s")
        
        results.append({
            'Model': name,
            'Accuracy': accuracy,
            'F1-Macro': f1_macro,
            'F1-Weighted': f1_weighted,
            'CV Score': cv_scores.mean(),
            'CV Std': cv_scores.std(),
            'Train Time': train_time,
            'Predict Time': predict_time,
            'model': model  # Store the model object
        })
        
        # Track best model based on F1-Macro
        if f1_macro > best_f1:
            best_f1 = f1_macro
            best_model = model
            best_model_name = name
        
        # Show detailed report for best models
        if name in ['Logistic Regression', 'Linear SVM (SGD)']:
            print(f"\nDetailed Classification Report:")
            print(classification_report(y_test, y_pred, target_names=le.classes_))
            
    except Exception as e:
        print(f"✗ Error: {str(e)}")

# Results summary
if results:
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('F1-Macro', ascending=False)
    
    print("\n" + "="*60)
    print("MODEL RANKING (by F1-Macro)")
    print("="*60)
    print(results_df[['Model', 'F1-Macro', 'Accuracy', 'CV Score', 'Train Time']].to_string(
        index=False, float_format='%.4f'
    ))
    
    print("\n" + "="*60)
    print("RECOMMENDATION")
    print("="*60)
    best = results_df.iloc[0]
    print(f"Best Model: {best['Model']}")
    print(f"F1-Macro: {best['F1-Macro']:.4f}")
    print(f"Accuracy: {best['Accuracy']:.4f}")
    print(f"Training Time: {best['Train Time']:.2f} seconds")
    
    # ============================================
    # SAVE THE BEST MODEL (FIXED)
    # ============================================
    print("\n" + "="*60)
    print("SAVING BEST MODEL")
    print("="*60)
    
    # Save the best model (from tracking)
    if best_model is not None:
        joblib.dump(best_model, 'best_model.pkl')
        print(f"✅ Best model ({best_model_name}) saved as 'best_model.pkl'")
        
        # Also save the best model with its name
        joblib.dump(best_model, f'{best_model_name.lower().replace(" ", "_")}_model.pkl')
        print(f"✅ Model also saved as '{best_model_name.lower().replace(' ', '_')}_model.pkl'")
    else:
        print("⚠️ No best model found to save")
    
    # Save other necessary files if they exist
    if 'vectorizer' in locals():
        joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
        print("✅ tfidf_vectorizer.pkl saved")
    
    if 'le' in locals():
        joblib.dump(le, 'label_encoder.pkl')
        print("✅ label_encoder.pkl saved")
    
    # Save results for reference
    results_df.to_csv('model_comparison_results.csv', index=False)
    print("✅ Results saved to 'model_comparison_results.csv'")
    
else:
    print("\n❌ No models were successfully trained.")

print("\n" + "="*60)
print("PROCESS COMPLETE")
print("="*60)


MODEL COMPARISON FOR SENTIMENT ANALYSIS
Training samples: 931,931
Test samples: 399,400
Features: 5,000
Classes: ['negative', 'neutral', 'positive']

Training: Logistic Regression
✓ Accuracy: 0.6059
✓ F1-Macro: 0.6020
✓ F1-Weighted: 0.6021
✓ CV Score: 0.6055 (+/- 0.0006)
✓ Train Time: 65.34s
✓ Predict Time: 0.14s

Detailed Classification Report:
              precision    recall  f1-score   support

    negative       0.51      0.82      0.63    133142
     neutral       0.65      0.45      0.53    132793
    positive       0.78      0.55      0.65    133465

    accuracy                           0.61    399400
   macro avg       0.65      0.61      0.60    399400
weighted avg       0.65      0.61      0.60    399400


Training: Linear SVM (SGD)
✓ Accuracy: 0.5846
✓ F1-Macro: 0.5740
✓ F1-Weighted: 0.5742
✓ CV Score: 0.5847 (+/- 0.0012)
✓ Train Time: 9.35s
✓ Predict Time: 0.10s

Detailed Classification Report:
              precision    recall  f1-score   support

    negative       0

In [20]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import warnings
import joblib

warnings.filterwarnings('ignore')

print("="*70)
print("Grid Search CV and Randomized Search Compairaison")
print("="*70)

# Optimization - 1
search_size = int(0.1 * X_train.shape[0]) 
X_train_sample = X_train[:search_size]
y_train_sample = y_train[:search_size]


# Grid Search CV

param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1','l2'],
    'solver': ['saga'],
    'class_weight': [None, 'balanced'],
    'max_iter': [1000]
}

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Parameter grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")
print(f"\nTotal combinations to try: {total_combinations}")


# Training
st_time = time.time()

gridSearchCV = GridSearchCV(
    LogisticRegression(random_state = 42),
    param_grid,
    cv = 3,
    scoring = 'f1_macro',
    n_jobs = -1,
    verbose = 1
)
gridSearchCV.fit(X_train_sample, y_train_sample)

grid_time = time.time() - st_time

print(f"\nGrid Search Completed in: {grid_time:.2f} seconds")
print("Best Parameter: \n")
for param, values in gridSearchCV.best_params_.items():
    print(f"  {param}: {values}")

print(f"Best Cross-validation F1 score: {gridSearchCV.best_score_:.4f}")

# --- FINAL STEP: Retrain on FULL dataset ---
print("\nRetraining best model on full 500k rows...")
best_model = gridSearchCV.best_estimator_
best_model.fit(X_train, y_train)

# --- SAVE THE MODEL ---
model_filename = 'best_logistic_model.pkl'
joblib.dump(best_model, model_filename)
print(f"Model saved as {model_filename}")


#Evaluation
y_pred_grid = best_model.predict(X_test)
accuracy_grid = accuracy_score(y_test,y_pred_grid)
f1_grid = f1_score(y_test,y_pred_grid, average = 'macro')

print(f"\nTest Set Performance:")
print(f"  Accuracy: {accuracy_grid:.4f}")
print(f"  F1-Macro: {f1_grid:.4f}")

Grid Search CV and Randomized Search Compairaison
Parameter grid:
  C: [0.01, 0.1, 1.0, 10.0]
  penalty: ['l1', 'l2']
  solver: ['saga']
  class_weight: [None, 'balanced']
  max_iter: [1000]

Total combinations to try: 16
Fitting 3 folds for each of 16 candidates, totalling 48 fits

Grid Search Completed in: 1111.88 seconds
Best Parameter: 

  C: 1.0
  class_weight: balanced
  max_iter: 1000
  penalty: l2
  solver: saga
Best Cross-validation F1 score: 0.5838

Retraining best model on full 500k rows...
Model saved as best_logistic_model.pkl

Test Set Performance:
  Accuracy: 0.6063
  F1-Macro: 0.6025


In [22]:
# Randomized Search

param_dist = {
    'C': np.logspace(-3,2,100),
    'penalty': ['l1','l2','elasticnet'],
    'solver': ['liblinear','saga','lbfgs','newton-cg'],
    'class_weight': [None,'balanced'],
    'max_iter': [1000,2000,3000],
    'tol': [1e-5, 1e-4, 1e-3, 1e-2]
}

print(f"Parameter distribution:")
for param, values in param_dist.items():
    if isinstance(values, np.ndarray):
        print(f"  {param}: {len(values)} values (range: {values[0]:.1e} to {values[-1]:.1e})")
    elif isinstance(values, list):
        print(f"  {param}: {values}")
    else:
        print(f"  {param}: {values}")


# Training
st_time = time.time()

random_search = RandomizedSearchCV(
    LogisticRegression(random_state = 42),
    param_dist,
    n_iter = 50,
    cv = 5,
    scoring = 'f1_macro',
    n_jobs = -1,
    verbose = 1,
    random_state = 42
)

random_search.fit(X_train_sample,y_train_sample)

random_search_time = time.time() - st_time

print(f"Randomized Search Completed in {random_search_time:.2f}")
print("\nBest Parameters:\n")
for param,value in random_search.best_params_.items():
    print(f"{param},{value}")
print(f"\nBest Cross-Validation F1-Macro: {random_search.best_score_:.4f}")


# --- FINAL STEP: Retrain on FULL dataset ---
print("\nRetraining best model on full 500k rows...")
best_model_2 = random_search.best_estimator_
best_model_2.fit(X_train, y_train)

# --- SAVE THE MODEL ---
model_filename = 'best_logistic_model_2.pkl'
joblib.dump(best_model_2, model_filename)
print(f"Model saved as {model_filename}")


#Evaluation:
y_pred_random = random_search.best_estimator_.predict(X_test)
random_search_accuracy = accuracy_score(y_test,y_pred_random)
f1_random = f1_score(y_test,y_pred_random,average = 'macro')

print(f"\nTest Set Performance:")
print(f"  Accuracy: {accuracy_random:.4f}")
print(f"  F1-Macro: {f1_random:.4f}")

Parameter distribution:
  C: 100 values (range: 1.0e-03 to 1.0e+02)
  penalty: ['l1', 'l2', 'elasticnet']
  solver: ['liblinear', 'saga', 'lbfgs', 'newton-cg']
  class_weight: [None, 'balanced']
  max_iter: [1000, 2000, 3000]
  tol: [1e-05, 0.0001, 0.001, 0.01]
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Randomized Search Completed in 1516.42

Best Parameters:

tol,0.001
solver,liblinear
penalty,l2
max_iter,1000
class_weight,None
C,4.862601580065354

Best Cross-Validation F1-Macro: 0.5827

Retraining best model on full 500k rows...
Model saved as best_logistic_model_2.pkl

Test Set Performance:


NameError: name 'accuracy_random' is not defined

In [23]:
#Evaluation:
y_pred_random = random_search.best_estimator_.predict(X_test)
random_search_accuracy = accuracy_score(y_test,y_pred_random)
f1_random = f1_score(y_test,y_pred_random,average = 'macro')

print(f"\nTest Set Performance:")
print(f"  Accuracy: {random_search_accuracy:.4f}")
print(f"  F1-Macro: {f1_random:.4f}")


Test Set Performance:
  Accuracy: 0.6041
  F1-Macro: 0.5995
